# 01 — Préparation des données pour Tableau

**Projet :** Tableau de bord de performance d'une place de marché en ligne (Olist)
**Auteure :** Danielle OUATTARA
**Données :** [Brazilian E-Commerce Public Dataset by Olist — Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

**Objectif :** transformer les 9 tables d'Olist en **deux fichiers simples**, directement exploitables dans Tableau Public :

| Fichier | Une ligne par… | Sert à… |
|---|---|---|
| `outputs/tableau_commandes.csv` | commande | croissance, géographie, livraison, satisfaction, fidélité |
| `outputs/tableau_articles.csv` | article vendu | catégories de produits |

| Étape | Contenu |
|---|---|
| 1 | Import des tables |
| 2 | Contrôles de qualité |
| 3 | Périmètre : commandes livrées, mois complets |
| 4 | Indicateurs par commande |
| 5 | Fidélité : rang de la commande pour chaque client |
| 6 | Table des articles |
| 7 | Contrôles finaux et export |
| 8 | Premiers chiffres clés |

## 1. Import des tables

In [1]:
import pandas as pd                                   # manipulation de tableaux
import numpy as np                                    # calculs numériques

pd.set_option("display.max_columns", None)            # afficher toutes les colonnes
DOSSIER = "../data/"                                  # dossier où se trouvent les 9 fichiers Olist

dates_commandes = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                   "order_delivered_customer_date", "order_estimated_delivery_date"]   # colonnes à lire comme des dates

commandes = pd.read_csv(DOSSIER + "olist_orders_dataset.csv", parse_dates=dates_commandes)  # commandes et dates clés
clients = pd.read_csv(DOSSIER + "olist_customers_dataset.csv")                             # clients et localisation
articles = pd.read_csv(DOSSIER + "olist_order_items_dataset.csv")                          # articles de chaque commande
paiements = pd.read_csv(DOSSIER + "olist_order_payments_dataset.csv")                      # paiements
avis = pd.read_csv(DOSSIER + "olist_order_reviews_dataset.csv")                            # notes laissées par les clients
produits = pd.read_csv(DOSSIER + "olist_products_dataset.csv")                             # caractéristiques des produits
vendeurs = pd.read_csv(DOSSIER + "olist_sellers_dataset.csv")                              # vendeurs
traduction = pd.read_csv(DOSSIER + "product_category_name_translation.csv")               # catégories en anglais

for nom, table in [("commandes", commandes), ("clients", clients), ("articles", articles),
                   ("paiements", paiements), ("avis", avis), ("produits", produits), ("vendeurs", vendeurs)]:
    print(f"{nom:<10} {table.shape}")                  # nombre de lignes et de colonnes de chaque table

commandes  (99441, 8)
clients    (99441, 5)
articles   (112650, 7)
paiements  (103886, 5)
avis       (99224, 7)
produits   (32951, 9)
vendeurs   (3095, 4)


## 2. Contrôles de qualité

In [2]:
print(commandes["order_status"].value_counts())        # statut des commandes : la grande majorité doit être « delivered »

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [3]:
print(f"Commandes en double : {commandes['order_id'].duplicated().sum()}")             # on attend 0
print(f"Commandes avec plusieurs avis : {avis['order_id'].duplicated().sum()}")        # quelques commandes ont 2 avis ou plus
print(f"Produits sans catégorie : {produits['product_category_name'].isna().sum()}")   # environ 600 produits

Commandes en double : 0
Commandes avec plusieurs avis : 551
Produits sans catégorie : 610


In [4]:
commandes.groupby(commandes["order_purchase_timestamp"].dt.to_period("M"))["order_id"].count()   # nombre de commandes par mois

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: order_id, dtype: int64

Le tableau mensuel montre que les premiers mois (fin 2016) et les derniers (septembre-octobre 2018) contiennent très peu de commandes : la collecte y est incomplète. Les garder ferait croire à un effondrement artificiel de l'activité en fin de période.

**Choix :** conserver les commandes livrées, achetées entre le 1er janvier 2017 et le 31 août 2018 (20 mois complets).

## 3. Périmètre : commandes livrées, mois complets

In [5]:
debut, fin = "2017-01-01", "2018-08-31 23:59:59"                                    # bornes de la période retenue
livrees = commandes[(commandes["order_status"] == "delivered")                       # commandes livrées
                    & commandes["order_purchase_timestamp"].between(debut, fin)      # achetées dans la période
                    & commandes["order_delivered_customer_date"].notna()].copy()     # avec une date de livraison connue
print(f"Commandes retenues : {len(livrees):,} sur {len(commandes):,}")

Commandes retenues : 96,203 sur 99,441


## 4. Indicateurs par commande

In [6]:
montants = (articles.groupby("order_id")                                  # un groupe par commande
            .agg(nb_articles=("order_item_id", "count"),                  # nombre d'articles
                 chiffre_affaires=("price", "sum"),                       # somme des prix des articles
                 frais_port=("freight_value", "sum"))                     # somme des frais de port
            .reset_index())

In [7]:
paiement_principal = (paiements.sort_values("payment_value", ascending=False)     # le paiement le plus élevé en premier
                      .drop_duplicates("order_id")                                # on garde ce paiement principal
                      [["order_id", "payment_type", "payment_installments"]])     # moyen de paiement et nombre d'échéances
paiement_principal["payment_type"] = paiement_principal["payment_type"].map(     # traduction des moyens de paiement
    {"credit_card": "Carte bancaire", "boleto": "Boleto (paiement en espèces)",
     "voucher": "Bon d'achat", "debit_card": "Carte de débit"}).fillna("Autre")

In [8]:
note = (avis.sort_values("review_answer_timestamp")                       # si plusieurs avis, on garde le plus récent
        .drop_duplicates("order_id", keep="last")
        [["order_id", "review_score"]])

In [9]:
categories = produits.merge(traduction, on="product_category_name", how="left")   # ajoute le nom anglais de chaque catégorie
categories["categorie"] = (categories["product_category_name_english"]
                           .fillna(categories["product_category_name"])          # sinon, le nom portugais
                           .fillna("non_renseignee")                              # sinon, « non renseignée »
                           .str.replace("_", " ").str.capitalize())               # « health_beauty » devient « Health beauty »

art_cat = articles.merge(categories[["product_id", "categorie"]], on="product_id", how="left")
categorie_principale = (art_cat.groupby(["order_id", "categorie"])["price"].sum()  # chiffre d'affaires par commande et catégorie
                        .reset_index()
                        .sort_values("price", ascending=False)
                        .drop_duplicates("order_id")                             # la catégorie qui pèse le plus dans la commande
                        .rename(columns={"categorie": "categorie_principale"})
                        [["order_id", "categorie_principale"]])

In [10]:
etats = {"AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceará",
         "DF": "Distrito Federal", "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso",
         "MS": "Mato Grosso do Sul", "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná",
         "PE": "Pernambuco", "PI": "Piauí", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
         "RS": "Rio Grande do Sul", "RO": "Rondônia", "RR": "Roraima", "SC": "Santa Catarina",
         "SP": "São Paulo", "SE": "Sergipe", "TO": "Tocantins"}            # noms complets : Tableau les reconnaît sur la carte

base = (livrees.merge(clients, on="customer_id", how="left")             # ajoute le client et sa localisation
               .merge(montants, on="order_id", how="left")              # ajoute les montants
               .merge(paiement_principal, on="order_id", how="left")    # ajoute le paiement principal
               .merge(note, on="order_id", how="left")                  # ajoute la note
               .merge(categorie_principale, on="order_id", how="left")) # ajoute la catégorie principale
base["etat"] = base["customer_state"].map(etats)                       # nom complet de l'État du client

In [11]:
achat = base["order_purchase_timestamp"]                                                   # date d'achat
base["date_achat"] = achat.dt.date                                                         # date sans l'heure
base["delai_livraison_jours"] = (base["order_delivered_customer_date"] - achat).dt.days   # jours entre achat et livraison
base["retard_jours"] = (base["order_delivered_customer_date"].dt.normalize()               # jours de retard par rapport
                        - base["order_estimated_delivery_date"].dt.normalize()).dt.days    # à la date estimée (négatif = en avance)
base["en_retard"] = (base["retard_jours"] > 0).astype(int)                                 # 1 si livrée après la date estimée
base["tranche_retard"] = pd.cut(base["retard_jours"],                                      # tranches lisibles pour le dashboard
                                bins=[-1000, 0, 3, 7, 14, 1000],
                                labels=["À l'heure ou en avance", "1 à 3 jours", "4 à 7 jours",
                                        "8 à 14 jours", "Plus de 14 jours"])
base["note_negative"] = (base["review_score"] <= 2).astype(int)                            # 1 si note de 1 ou 2 sur 5

## 5. Fidélité : rang de la commande pour chaque client

Dans Olist, `customer_id` change à chaque commande : il faut utiliser `customer_unique_id` pour reconnaître un même client.

In [12]:
base = base.sort_values("order_purchase_timestamp")                                   # ordre chronologique
base["rang_commande"] = base.groupby("customer_unique_id").cumcount() + 1             # 1 = première commande du client
base["nb_commandes_client"] = base.groupby("customer_unique_id")["order_id"].transform("count")   # total de commandes du client
base["client_recurrent"] = (base["nb_commandes_client"] >= 2).astype(int)              # 1 si le client a commandé au moins deux fois

## 6. Table des articles

In [13]:
table_articles = (art_cat[art_cat["order_id"].isin(base["order_id"])]                 # articles des commandes retenues
                  .merge(vendeurs[["seller_id", "seller_state"]], on="seller_id", how="left")
                  .merge(base[["order_id", "date_achat", "etat", "review_score", "en_retard"]], on="order_id", how="left")
                  .rename(columns={"price": "prix", "freight_value": "frais_port", "seller_state": "etat_vendeur"})
                  [["order_id", "product_id", "seller_id", "categorie", "prix", "frais_port",
                    "etat_vendeur", "date_achat", "etat", "review_score", "en_retard"]])
print(table_articles.shape)

(109872, 11)


## 7. Contrôles finaux et export

In [14]:
colonnes = {"order_id": "commande", "customer_unique_id": "client", "date_achat": "date_achat",
            "etat": "etat_client", "customer_city": "ville_client", "nb_articles": "nb_articles",
            "chiffre_affaires": "chiffre_affaires", "frais_port": "frais_port",
            "payment_type": "moyen_paiement", "payment_installments": "nb_echeances",
            "categorie_principale": "categorie_principale", "review_score": "note",
            "note_negative": "note_negative", "delai_livraison_jours": "delai_livraison_jours",
            "retard_jours": "retard_jours", "en_retard": "en_retard", "tranche_retard": "tranche_retard",
            "rang_commande": "rang_commande", "client_recurrent": "client_recurrent"}   # noms clairs, en français
table_commandes = base[list(colonnes)].rename(columns=colonnes)

print(table_commandes.isna().sum()[table_commandes.isna().sum() > 0])   # valeurs manquantes restantes (quelques notes, montants)

note    643
dtype: int64


In [15]:
table_commandes.to_csv("../outputs/tableau_commandes.csv", index=False)   # une ligne par commande
table_articles.to_csv("../outputs/tableau_articles.csv", index=False)     # une ligne par article vendu
print("Export terminé : outputs/tableau_commandes.csv et outputs/tableau_articles.csv")

Export terminé : outputs/tableau_commandes.csv et outputs/tableau_articles.csv


## 8. Premiers chiffres clés

Ces chiffres servent à vérifier le dashboard Tableau : il doit afficher exactement les mêmes.

In [16]:
tc = table_commandes
print(f"Commandes             : {len(tc):,}")
print(f"Clients uniques       : {tc['client'].nunique():,}")
print(f"Chiffre d'affaires    : {tc['chiffre_affaires'].sum():,.0f} R$")
print(f"Panier moyen          : {tc['chiffre_affaires'].mean():,.1f} R$")
print(f"Délai moyen           : {tc['delai_livraison_jours'].mean():.1f} jours")
print(f"Taux de retard        : {tc['en_retard'].mean():.1%}")
print(f"Note moyenne          : {tc['note'].mean():.2f} / 5")
print(f"Clients récurrents    : {tc.groupby('client')['client_recurrent'].max().mean():.1%}")

Commandes             : 96,203
Clients uniques       : 93,096
Chiffre d'affaires    : 13,179,778 R$
Panier moyen          : 137.0 R$
Délai moyen           : 12.1 jours
Taux de retard        : 6.8%
Note moyenne          : 4.16 / 5
Clients récurrents    : 3.0%


In [17]:
tc.groupby("tranche_retard", observed=True).agg(commandes=("commande", "count"),   # effet du retard sur la satisfaction
                                                note_moyenne=("note", "mean"),
                                                part_notes_negatives=("note_negative", "mean")).round(2)

,commandes,note_moyenne,part_notes_negatives
tranche_retard,,,
À l'heure ou en avance,89672,4.29,0.09
1 à 3 jours,1869,3.29,0.32
4 à 7 jours,1802,2.10,0.66
8 à 14 jours,1478,1.67,0.78
Plus de 14 jours,1382,1.72,0.76
